# ДЗ 3

In [36]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Задача 3

Вычислить:
$$S = \sum_{n=1}^{\infty} \frac{n^2 + 1}{n^4 + n^2 + 1}\cos(2n)$$

с точностью $\varepsilon = 10^{-6}$.

На сайте показано, что можно посчитать следующим образом:

$$B = 1 - \pi + \frac{\pi^2}{6}$$

$$
S = B + \sum_{n=1}^{\infty} \frac{-\cos(2n)}{n^2(n^4+n^2+1)},
$$

причем для нужной точности достаточно 10 или более слагаемых

In [37]:
B = 1.0 - np.pi + np.pi**2 / 6.0

eps = 1e-6

S_remainder = 0.0
for n in range(1, 10):
    term = -np.cos(2 * n) / (n**2 * (n**4 + n**2 + 1))
    S_remainder += term
    if n > 1 and abs(term) < eps:
        N_kummer = n
        break

S_kummer = B + S_remainder

n=0
S_direct = 0.0
while True:
    n += 1
    term = (n**2 + 1) / (n**4 + n**2 + 1) * np.cos(2 * n)
    S_direct += term
    if n > 1 and abs(term) < eps:
        N_direct = n
        break

print(f"Метод Куммера: S = {S_kummer:.10f},  членов ряда: {N_kummer}")
print(f"Прямое суммирование: S = {S_direct:.10f},  членов ряда: {N_direct}")
print(f"Разница: = {abs(S_kummer - S_direct):.2e}")

Метод Куммера: S = -0.3512653599,  членов ряда: 18
Прямое суммирование: S = -0.3512801362,  членов ряда: 150
Разница: = 1.48e-05


## Задача 4

$$\lim_{x \to 0} \frac{y(x) - 1}{x}, \quad y(x) = \begin{cases} \cosh\sqrt{x}, & x > 0 \\ \cos\sqrt{|x|}, & x < 0 \end{cases}$$

### Аналитическое значение

При $x > 0$: $\cosh\sqrt{x} = 1 + \frac{x}{2} + \frac{x^2}{24} + \ldots \Rightarrow \frac{\cosh\sqrt{x} - 1}{x} = \frac{1}{2} + \frac{x}{24} + \ldots \to \frac{1}{2}$.

При $x < 0$: $\cos\sqrt{|x|} = 1 - \frac{|x|}{2} + \frac{|x|^2}{24} - \ldots = 1 + \frac{x}{2} + \frac{x^2}{24} + \ldots \Rightarrow \frac{\cos\sqrt{|x|} - 1}{x} = \frac{1}{2} + \frac{x}{24} + \ldots \to \frac{1}{2}$.

Предел равен $\frac{1}{2}$.

### Метод из лекции: симметризация + экстраполяция на сетке

Обозначим $F(x) = (y(x) - 1)/x$. Симметризуем:
$$f(x) = \frac{F(x) + F(-x)}{2} = \frac{\cosh\sqrt{x} - \cos\sqrt{x}}{2x}, \quad x > 0$$

Получаем односторонний предел $x \to 0^+$. Тогда т.к. $\sqrt{x}$ вещественный сделаем замену $x = t^2$. Получаем двусторонний предел от
$$f(t^2) = \frac{\cosh(t) - \cos(t)}{2t^2} - \text{четная и аналитическая}$$

Далее по алгоритму строим сетку и т.д.

In [38]:
from math import comb, factorial

def f_symmetric(t):
    if abs(t) < 1e-30:
        return 0.5
    return (np.cosh(t) - np.cos(t)) / (2.0 * t**2)

def compute_D(n, j_max):
    D = [0.0] * (j_max + 1)
    D[0] = 1.0
    for j in range(j_max):
        val = 0.0
        p = n + 2 * j + 2
        for k in range(n + 2 * j + 1):
            for l in range(j + 1):
                bi = k - j + l
                bn = n + 2 * l
                if bi < 0 or bi > bn:
                    continue
                base = n + 2 * j - 2 * k
                val += (-1)**k * D[l] * comb(bn, bi) * base**p / (2**p * factorial(p))
        D[j + 1] = val
    return D

def compute_A(k, n, m_val):
    D = compute_D(n, m_val)
    val = 0.0
    for l in range(m_val + 1):
        bi = k - m_val + l
        bn = n + 2 * l
        if bi < 0 or bi > bn:
            continue
        val += (-1)**(k - m_val) * D[l] * comb(bn, bi)
    return val

def compute_limit(f_even, dx, m, N=None):
    if N is None:
        N = 2 * m + 1
    j0 = 2 * m + 2

    y = {}
    for k in range(j0):
        v = f_even((2 * k + 1) * dx / 2.0)
        y[k] = v
        y[-(k + 1)] = v

    result = 0.0
    for n in range(N + 1):
        nmod2 = n % 2
        upper_k = 2 * m + nmod2
        m_val = m - (n - nmod2) // 2
        coeff = (-1)**n / (4**n * factorial(n))
        for k in range(upper_k + 1):
            A_val = compute_A(k, n, m_val)
            y_idx = 2 * m + nmod2 - 2 * k
            result += A_val * coeff * y[y_idx]
    return result

exact = 0.5
m = 3

print(f"Порядок схемы m = {m}, число точек j0 = {2*m+2}")
print()
for dx in [2.0, 1.0, 0.5, 0.1, 0.01]:
    val = compute_limit(f_symmetric, dx, m)
    print(f"шаг = {dx} — предел = {val:.15f}")

Порядок схемы m = 3, число точек j0 = 8

шаг = 2.0 — предел = 0.902931007980118
шаг = 1.0 — предел = 0.500191957752235
шаг = 0.5 — предел = 0.500000579088194
шаг = 0.1 — предел = 0.500000000001456
шаг = 0.01 — предел = 0.499999999997539
